In [0]:
%run ../../config/utils

In [0]:
import pyspark.sql.functions as f
import pandas as pd
import numpy as np
from datetime import datetime

In [0]:
run_date_str = dbutils.widgets.get('run_as_date')
muhh_path = dbutils.widgets.get('muhh_path')
current_year = int(run_date_str[:4])

In [0]:
MUHH = spark.read.csv(muhh_path,header=True).withColumn('MBRSHP_SID', f.col('MBRSHP_SID').cast('long'))

prev_dt = (
    spark.table(gm_scores).filter(f.col("score_date") <= run_date_str)
      .agg(f.max("score_date").alias("prev_date"))
      .first()["prev_date"]
)
score_date = datetime.strftime(prev_dt,'%Y-%m-%d')
scored_file = spark.table(gm_scores).filter(f.col("score_date") == prev_dt)

In [0]:
combined = MUHH.select('MBRSHP_SID').distinct().join(scored_file,['MBRSHP_SID'],'inner')
print(combined.count())

# Top 6 Deciles

In [0]:
combined_1_6 = combined.filter(f.col('decile').isin(1,2,3,4,5,6))
combined_1_6.groupby('decile').agg(f.countDistinct('MBRSHP_SID')).show()
combined_1_6_pd = combined_1_6.toPandas()

# Circulation Count

In [0]:
Holdout_ct = 120_000
Holdout_Mailer1_Ct = 15_000
Holdout_Mailer2_Ct = 15_000
Holdout_Mailer3_Ct = 15_000

Mailer1_ct = 2_425_000
Mailer2_ct = 3_795_000
Mailer3_ct = 3_350_500

# Holdout

In [0]:
int(Holdout_ct/6)

In [0]:
Holdout_decile1 = combined_1_6_pd[combined_1_6_pd['decile']==1].sample(n = int(Holdout_ct/6))
Holdout_decile2 = combined_1_6_pd[combined_1_6_pd['decile']==2].sample(n = int(Holdout_ct/6))
Holdout_decile3 = combined_1_6_pd[combined_1_6_pd['decile']==3].sample(n = int(Holdout_ct/6))
Holdout_decile4 = combined_1_6_pd[combined_1_6_pd['decile']==4].sample(n = int(Holdout_ct/6))
Holdout_decile5 = combined_1_6_pd[combined_1_6_pd['decile']==5].sample(n = int(Holdout_ct/6))
Holdout_decile6 = combined_1_6_pd[combined_1_6_pd['decile']==6].sample(n = int(Holdout_ct/6))

Holdout = pd.concat([Holdout_decile1,Holdout_decile2,Holdout_decile3,Holdout_decile4,Holdout_decile5,Holdout_decile6])
Holdout.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

## Holdout - Mailer 1

In [0]:
data = combined_1_6_pd[~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout['MBRSHP_SID']))]

Holdout_Mailer1_decile1 = data[data['decile']==1].sample(n = int(Holdout_Mailer1_Ct/6))
Holdout_Mailer1_decile2 = data[data['decile']==2].sample(n = int(Holdout_Mailer1_Ct/6))
Holdout_Mailer1_decile3 = data[data['decile']==3].sample(n = int(Holdout_Mailer1_Ct/6))
Holdout_Mailer1_decile4 = data[data['decile']==4].sample(n = int(Holdout_Mailer1_Ct/6))
Holdout_Mailer1_decile5 = data[data['decile']==5].sample(n = int(Holdout_Mailer1_Ct/6))
Holdout_Mailer1_decile6 = data[data['decile']==6].sample(n = int(Holdout_Mailer1_Ct/6))

Holdout_Mailer1 = pd.concat([Holdout_Mailer1_decile1,
                     Holdout_Mailer1_decile2,
                     Holdout_Mailer1_decile3,
                     Holdout_Mailer1_decile4,
                     Holdout_Mailer1_decile5,
                     Holdout_Mailer1_decile6])

Holdout_Mailer1.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

## Holdout - Mailer 2

In [0]:
data = combined_1_6_pd[(~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout['MBRSHP_SID']))) 
                                  & (~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout_Mailer1['MBRSHP_SID'])))]

Holdout_Mailer2_decile1 = data[data['decile']==1].sample(n = int(Holdout_Mailer2_Ct/6))
Holdout_Mailer2_decile2 = data[data['decile']==2].sample(n = int(Holdout_Mailer2_Ct/6))
Holdout_Mailer2_decile3 = data[data['decile']==3].sample(n = int(Holdout_Mailer2_Ct/6))
Holdout_Mailer2_decile4 = data[data['decile']==4].sample(n = int(Holdout_Mailer2_Ct/6))
Holdout_Mailer2_decile5 = data[data['decile']==5].sample(n = int(Holdout_Mailer2_Ct/6))
Holdout_Mailer2_decile6 = data[data['decile']==6].sample(n = int(Holdout_Mailer2_Ct/6))

Holdout_Mailer2 = pd.concat([Holdout_Mailer2_decile1,
                     Holdout_Mailer2_decile2,
                     Holdout_Mailer2_decile3,
                     Holdout_Mailer2_decile4,
                     Holdout_Mailer2_decile5,
                     Holdout_Mailer2_decile6])

Holdout_Mailer2.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

## Holdout - Mailer 3

In [0]:
data = combined_1_6_pd[(~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout['MBRSHP_SID']))) 
                                  & (~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout_Mailer1['MBRSHP_SID']))) 
                                  & (~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout_Mailer2['MBRSHP_SID'])))]

Holdout_Mailer3_decile1 = data[data['decile']==1].sample(n = int(Holdout_Mailer3_Ct/6))
Holdout_Mailer3_decile2 = data[data['decile']==2].sample(n = int(Holdout_Mailer3_Ct/6))
Holdout_Mailer3_decile3 = data[data['decile']==3].sample(n = int(Holdout_Mailer3_Ct/6))
Holdout_Mailer3_decile4 = data[data['decile']==4].sample(n = int(Holdout_Mailer3_Ct/6))
Holdout_Mailer3_decile5 = data[data['decile']==5].sample(n = int(Holdout_Mailer3_Ct/6))
Holdout_Mailer3_decile6 = data[data['decile']==6].sample(n = int(Holdout_Mailer3_Ct/6))

Holdout_Mailer3 = pd.concat([Holdout_Mailer3_decile1,
                     Holdout_Mailer3_decile2,
                     Holdout_Mailer3_decile3,
                     Holdout_Mailer3_decile4,
                     Holdout_Mailer3_decile5,
                     Holdout_Mailer3_decile6])

Holdout_Mailer3.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

## Mailer 2

### Deciles 1,2,3

In [0]:
Mailer2_123 = combined_1_6_pd[(~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout['MBRSHP_SID'])))
                          & (~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout_Mailer1['MBRSHP_SID'])))
                          & (~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout_Mailer2['MBRSHP_SID'])))
                          & (~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout_Mailer3['MBRSHP_SID'])))
                          & (combined_1_6_pd['decile'].isin([1,2,3]))]
Mailer2_123.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

### Deciles 4,5,6

In [0]:
data = combined_1_6_pd[(~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout['MBRSHP_SID'])))
                          & (~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout_Mailer1['MBRSHP_SID'])))
                          & (~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout_Mailer2['MBRSHP_SID'])))
                          & (~combined_1_6_pd['MBRSHP_SID'].isin(list(Holdout_Mailer3['MBRSHP_SID'])))
                          & (combined_1_6_pd['decile'].isin([4,5,6]))]

Mailer2_decile4 = data[data['decile']==4].sample(n = int((Mailer2_ct - len(Mailer2_123))/3) - int(Holdout_Mailer2_Ct/3))
Mailer2_decile5 = data[data['decile']==5].sample(n = int((Mailer2_ct - len(Mailer2_123))/3) - int(Holdout_Mailer2_Ct/3))
Mailer2_decile6 = data[data['decile']==6].sample(n = int((Mailer2_ct - len(Mailer2_123))/3) - int(Holdout_Mailer2_Ct/3))

Mailer2_456 = pd.concat([Mailer2_decile4,
                        Mailer2_decile5,
                        Mailer2_decile6])

Mailer2_456.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

In [0]:
Mailer2 = pd.concat([Mailer2_123,
                    Mailer2_456,
                    #Mailer 2 individual group
                    Holdout_Mailer2
                    ])

Mailer2.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

## Mailer 3

### Deciles 1,2,3

In [0]:
Mailer3_123 = Mailer2_123
Mailer3_123.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

### Deciles 4,5,6

In [0]:
Mailer3_decile4 = Mailer2_456[Mailer2_456['decile'] == 4].sample(n = int((Mailer3_ct - Holdout_Mailer3_Ct - len(Mailer3_123))/3))
Mailer3_decile5 = Mailer2_456[Mailer2_456['decile'] == 5].sample(n = int((Mailer3_ct - Holdout_Mailer3_Ct - len(Mailer3_123))/3))
Mailer3_decile6 = Mailer2_456[Mailer2_456['decile'] == 6].sample(n = int((Mailer3_ct - Holdout_Mailer3_Ct - len(Mailer3_123))/3))

Mailer3_456 = pd.concat([Mailer3_decile4,
                        Mailer3_decile5,
                        Mailer3_decile6])

Mailer3_456.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

In [0]:
Mailer3 = pd.concat([Mailer3_123,
                    Mailer3_456,
                    #Mailer 3 individual group
                    Holdout_Mailer3
                    ])

Mailer3.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

## Mailer 1

### Deciles 1,2,3

In [0]:
Mailer1_123 = Mailer2_123
Mailer1_123.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

### Deciles 4,5,6

In [0]:
Mailer1_decile4 = Mailer3_456[Mailer3_456['decile'] == 4].sample(n = int((Mailer1_ct - Holdout_Mailer1_Ct - len(Mailer1_123))/3))
Mailer1_decile5 = Mailer3_456[Mailer3_456['decile'] == 5].sample(n = int((Mailer1_ct - Holdout_Mailer1_Ct - len(Mailer1_123))/3))
Mailer1_decile6 = Mailer3_456[Mailer3_456['decile'] == 6].sample(n = int((Mailer1_ct - Holdout_Mailer1_Ct - len(Mailer1_123))/3))

Mailer1_456 = pd.concat([Mailer1_decile4,
                        Mailer1_decile5,
                        Mailer1_decile6])

Mailer1_456.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

In [0]:
Mailer1 = pd.concat([Mailer1_123,
                    Mailer1_456,
                    #Mailer 1 individual group
                    Holdout_Mailer1
                    ])

Mailer1.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

In [0]:
Mailer1

#### Digital coupons for 50k in Mailer 1 decile 2 and 50k Mailer 1 decile 3
##### Ensure these members receive all three mailers

In [0]:
Members_receivingall3mailers = Mailer1[['MBRSHP_SID','decile']].merge(Mailer2[['MBRSHP_SID','decile']], left_on = ['MBRSHP_SID','decile'], right_on = ['MBRSHP_SID','decile'], how = 'inner').merge(Mailer3[['MBRSHP_SID','decile']], left_on = ['MBRSHP_SID','decile'], right_on = ['MBRSHP_SID','decile'], how = 'inner')
Members_receivingall3mailers.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

In [0]:
Mailer1_Decile2_digital = Members_receivingall3mailers[Members_receivingall3mailers['decile'] == 2].sample(n = 50_000)
Mailer1_Decile3_digital = Members_receivingall3mailers[Members_receivingall3mailers['decile'] == 3].sample(n = 50_000)
Mailer1_digital = pd.concat([Mailer1_Decile2_digital,
                    Mailer1_Decile3_digital,
                    ])

In [0]:
for df, out_type in zip([Mailer1, Mailer2, Mailer3, Holdout, Holdout_Mailer1, Holdout_Mailer2, Holdout_Mailer3], ['Mailer1', 'Mailer2', 'Mailer3', 'Holdout','Holdout_Mailer1', 'Holdout_Mailer2', 'Holdout_Mailer3']):
    print(f'\n{out_type}')
    print(df.info())
    print(df.MBRSHP_SID.nunique())
    df['output_type'] = out_type
    spark_df = (
        spark.createDataFrame(df)
            .withColumn('MBRSHP_SID', f.col('MBRSHP_SID').cast('long'))
            .withColumn('mbrshp_nbr', f.col('mbrshp_nbr').cast('long'))
            .withColumn('score', f.col('score').cast('decimal(5,4)'))
            .withColumn('decile', f.col('decile').cast('integer'))
            .withColumn('score_date', f.col('score_date').cast('date'))
            .withColumn('output_type', f.col('output_type').cast('string'))
    )   
    spark_df.write.mode('overwrite').option('replaceWhere',f"score_date = '{score_date}' AND output_type = '{out_type}'").saveAsTable(gm_holdout_and_mailer)
    

In [0]:
df_digital = (
    spark.createDataFrame(Mailer1_digital)
        .withColumn('MBRSHP_SID', f.col('MBRSHP_SID').cast('long'))
        .withColumn('decile', f.col('decile').cast('integer'))
        .withColumn('score_date', f.lit(score_date).cast('date'))
)
df_digital.write.mode('overwrite').option('replaceWhere',f"score_date = '{score_date}'").saveAsTable(gm_mailer_digital)